In [ ]:
conda create -n sti_spatrack python=3.10 -y
conda activate sti_spatrack
pip install spaTrack scanpy anndata pandas numpy scipy matplotlib

In [ ]:

from pathlib import Path
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import spaTrack as spt

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "spaTrack"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")

# spaTrack expects spatial coordinates in X_spatial.
adata.obsm["X_spatial"] = adata.obsm["spatial"].copy()

# Work from original counts, then perform standard preprocessing.
adata.X = adata.layers["counts"].copy()
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Generate provisional clusters to help define a starting state.
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=min(30, adata.n_vars - 1))
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="cluster")

print(adata.obs["cluster"].value_counts())

# Plot clusters before selecting the root.
xy = adata.obsm["X_spatial"]
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["cluster"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack: provisional clusters")
fig.colorbar(scat, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# Change this after inspecting the plot and cluster composition.
START_CLUSTER = "0"

if START_CLUSTER not in adata.obs["cluster"].astype(str).values:
    raise ValueError("START_CLUSTER not found.")

# The package supports automatic assessment of starting clusters.
adata = spt.assess_start_cluster(adata)
spt.assess_start_cluster_plot(adata)
plt.savefig(OUT / "02_start_cluster_entropy.png", dpi=200,
            bbox_inches="tight")
plt.show()

# Set the starting cells using the selected cluster.
start_cells = spt.set_start_cells(
    adata, select_way="cell_type", cell_type=START_CLUSTER
)

# Optimal-transport-based transition matrix.
adata.obsp["trans"] = spt.get_ot_matrix(
    adata,
    data_type="spatial",
    alpha1=0.5,
    alpha2=0.5,
)

# Pseudotime and spatial velocity field.
adata.obs["ptime"] = spt.get_ptime(adata, start_cells)

adata.uns["E_grid"], adata.uns["V_grid"] = spt.get_velocity(
    adata, basis="spatial", n_neigh_pos=30
)

# Save the outputs.
adata.obs.to_csv(OUT / "cell_results.csv")
adata.write_h5ad(OUT / "spatrack_results.h5ad")

transition = adata.obsp["trans"]
sp.save_npz(OUT / "transition_matrix.npz", transition)

# Plot pseudotime in tissue coordinates.
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["ptime"], s=3, cmap="viridis"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack pseudotime")
fig.colorbar(scat, ax=ax, label="Pseudotime")
fig.tight_layout()
fig.savefig(OUT / "03_pseudotime.png", dpi=200)
plt.show()

# Plot the inferred spatial vector field.
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(xy[:, 0], xy[:, 1], c="lightgray", s=2, alpha=0.5)

E = adata.uns["E_grid"]
V = adata.uns["V_grid"]
ax.quiver(E[0], E[1], V[0], V[1], color="red", angles="xy")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack inferred spatial vector field")
fig.tight_layout()
fig.savefig(OUT / "04_vector_field.png", dpi=200)
plt.show()

print("spaTrack results saved to:", OUT)
